<a href="https://colab.research.google.com/github/ejr-00/Lab-4-LLM-s/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support



In [1]:
!pip install -q openai python-dotenv pandas matplotlib

In [3]:
import os
from google.colab import userdata
from openai import OpenAI

API_KEY = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [4]:
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content, response.usage


# Test the function
answer, usage = ask_llm("What is artificial intelligence?")

print("Response:")
print(answer)

print("\nToken usage:")
print(usage)

Response:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data and improve their performance over time.
2. **Reasoning**: AI systems can draw inferences, make decisions, and solve problems using logical rules and algorithms.
3. **Problem-solving**: AI systems can identify and solve complex problems, often using machine learning algorithms.
4. **Perception**: AI systems can interpret and understand data from sensors, such as images, speech, and text.
5. **Natural Language Processing (NLP)**: AI systems can understand, generate, and process human language.

AI systems use various techniques, including:

1. **Machine learning**: AI systems learn from data and improve their performance over time.
2. **Deep learning**: A subset of machine learning that uses neural networks to analyze data.
3. **Rule-based systems**: AI systems use pre-defined rule

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*

**Answer:**
The system role gives the model general instructions about how it should behave or what role it should take. For example, we can tell it to act as a helpful assistant or as an assistant to a microfinance loan officer. The user role contains the specific request or information that we want the model to respond to. For example, the user could ask the model to summarize a loan application.


*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** A token is a small unit of text that an LLM processes. It can represent a whole word, part of a word, punctuation, or another piece of text. API providers bill based on tokens because the amount of computation required depends more on how much text the model processes and generates than simply on the number of requests

### Part 1.2 — Temperature: the randomness dial

In [5]:
question = "Suggest a name for a savings product for market traders in Accra."

# Temperature = 0.0
print("===== TEMPERATURE 0.0 =====")

for i in range(5):
    answer, usage = ask_llm(
        question,
        temperature=0.0,
        max_tokens=100
    )
    print(f"\nAttempt {i+1}:")
    print(answer)


# Temperature = 1.2
print("\n===== TEMPERATURE 1.2 =====")

for i in range(5):
    answer, usage = ask_llm(
        question,
        temperature=1.2,
        max_tokens=100
    )
    print(f"\nAttempt {i+1}:")
    print(answer)

===== TEMPERATURE 0.0 =====

Attempt 1:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language

Attempt 2:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Market Fund**: This name is straightforward and clearly communicates the product's purpose and target audience.
4.

Attempt 3:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makol

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, the responses were generally more consistent because the model had very little randomness in its generation. At temperature 1.2, the responses showed more variation, with different names and ideas being suggested for the same question.

For the loan decision-support system, a lower temperature such as 0.0 is more appropriate for tasks such as extracting structured information because consistency and accuracy are more important than creativity. A higher temperature would be more appropriate for creative tasks where generating different ideas is useful.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [7]:

# SUMMARY PROMPT V1


SUMMARY_PROMPT_V1 = """
Summarize this loan application:

{letter_text}
"""

print("===== V1: L002 =====")
answer, _ = ask_llm(
    SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"]),
    temperature=0.7,
    max_tokens=200
)
print(answer)

print("\n===== V1: L006 =====")
answer, _ = ask_llm(
    SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"]),
    temperature=0.7,
    max_tokens=200
)
print(answer)



# SUMMARY PROMPT V2


SUMMARY_SYSTEM_V2 = """
You are an assistant to a microfinance loan officer.

Summarize loan applications into a short, factual and neutral brief.
Use ONLY information explicitly stated in the application.
Do not invent, assume, or infer missing information.
Keep the summary to 3-4 sentences.
Mention the applicant, requested amount, purpose of the loan,
relevant financial information, repayment information, and
collateral or guarantor information when available.
"""

SUMMARY_PROMPT_V2 = """
Summarize this loan application:

{letter_text}
"""

print("\n===== V2: L002 =====")
answer, _ = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0,
    max_tokens=200
)
print(answer)

print("\n===== V2: L006 =====")
answer, _ = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0,
    max_tokens=200
)
print(answer)

===== V1: L002 =====
Kwame Boateng, a commercial driver, is applying for a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He expects his business to improve after the festive season and is willing to repay the loan when his finances allow, but he currently has no collateral to offer. He is seeking urgent assistance with the loan.

===== V1: L006 =====
Here is a summary of Kofi's loan application:

* Loan amount: GHS 50,000
* Proposed businesses: car washing, provision shop, and importing phones from Dubai
* Applicant's age: 22
* Repayment plan: 1 year, relying on the expected success of the businesses
* Collateral: None, but Kofi claims to be trustworthy
* Experience: No prior experience in any of the proposed businesses, but friends consider him "business-minded"

===== V2: L002 =====
Kwame Boateng, a commercial driver, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that his business has been slow,

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** V1 gave a general summary without clearly controlling what information should be included. It could focus on the applicant's story while leaving out important loan information such as the requested amount, repayment terms, or collateral. V2 fixed this by explicitly requiring a factual and neutral 3–4 sentence summary containing the applicant, loan amount, purpose, financial information, repayment information, and collateral or guarantor information.

2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

"No invented" details is essential because a loan officer could make a financial decision based on information that was never provided by the applicant. When an LLM generates information that is not supported by its input, this failure mode is called a hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [8]:
import json
import pandas as pd


EXTRACT_SYSTEM_PROMPT = """
You are a data extraction assistant for a microfinance institution.

Extract information from the loan application and return ONLY valid JSON.

Use exactly these keys:
- applicant_name: string
- amount_ghs: number
- purpose: string
- monthly_profit_ghs: number or null
- has_collateral_or_guarantor: boolean
- repayment_months: number or null

Rules:
1. Use ONLY information explicitly stated in the letter.
2. If a field is not stated, use null.
3. Do not guess or infer missing information.
4. Return ONLY the JSON object.
5. Do not include markdown, explanations, or code fences.
"""


EXTRACT_PROMPT = """
Extract the required information from this loan application.

Example:

Letter:
"I am Ama Mensima. I request GHS 5,000 to purchase equipment for my bakery.
My monthly profit is GHS 700. My brother will guarantee the loan.
I will repay the loan over 10 months."

JSON:
{{
  "applicant_name": "Ama Mensima",
  "amount_ghs": 5000,
  "purpose": "purchase equipment for bakery",
  "monthly_profit_ghs": 700,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}}

Now extract the information from this letter:

{letter_text}
"""


def extract_fields(letter_text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": EXTRACT_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": EXTRACT_PROMPT.format(
                    letter_text=letter_text
                )
            }
        ],
        temperature=0.0,
        max_tokens=300
    )

    result = response.choices[0].message.content.strip()

    # Remove possible markdown JSON fences
    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    if result.endswith("```"):
        result = result[:-3]

    result = result.strip()

    try:
        return json.loads(result)

    except json.JSONDecodeError:
        print("WARNING: Could not parse model output as JSON.")
        print("Raw output:", result)
        return None


# Run extraction on all six letters
results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is None:
        extracted = {
            "applicant_name": None,
            "amount_ghs": None,
            "purpose": None,
            "monthly_profit_ghs": None,
            "has_collateral_or_guarantor": None,
            "repayment_months": None
        }

    extracted["letter_id"] = letter_id
    results.append(extracted)


extracted_df = pd.DataFrame(results)

# Put letter_id first
columns = [
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

extracted_df = extracted_df[columns]

display(extracted_df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers for poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*

**Answer:** The example should not come from the six letters because it could introduce information from the evaluation dataset into the prompt. Using an independent example tests whether the model can generalize the extraction instructions to new applications rather than simply reproducing information it has already seen.

*2. Why "use null, do not guess" — what did the model do without that instruction?*

**Answer:** The instruction is important because some applications do not provide all the required information. Without it, the model may try to fill missing fields using assumptions or information that is not actually in the letter. Using null makes it clear that the information is unavailable instead of presenting an invented value as fact.

*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** Temperature 0 is appropriate for extraction because we want consistent and predictable results from the same information. For creative tasks, higher temperature can be useful because it allows more variation and produces a wider range of ideas.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [9]:
BRIEF_SYSTEM_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Your job is to analyze a loan application and provide decision support.
The final loan decision must always be made by a human loan officer.

Use ONLY information contained in the application and extracted data.
Do not invent facts or make unsupported assumptions.

Your response must contain exactly these four sections:

1. Strengths
- List positive factors supported by the application.

2. Risks / Red Flags
- List financial, business, repayment, or other concerns supported by the application.

3. Missing Information
- List important information or documents the loan officer should request.

4. Suggested Next Step
- Recommend an appropriate action such as requesting documents,
  inviting the applicant for an interview, or flagging the application
  for senior review.
- NEVER say "approve" or "reject".
"""


BRIEF_PROMPT = """
Prepare a decision-support brief for this loan application.

ORIGINAL APPLICATION:
{letter_text}

EXTRACTED INFORMATION:
{extracted_data}

Remember:
- Ground every point in the available information.
- Do not invent facts.
- The final decision belongs to a human loan officer.
- Do not recommend approval or rejection.
"""


def generate_brief(letter_text, extracted_data):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": BRIEF_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": BRIEF_PROMPT.format(
                    letter_text=letter_text,
                    extracted_data=json.dumps(
                        extracted_data,
                        indent=2
                    )
                )
            }
        ],
        temperature=0.0,
        max_tokens=500
    )

    return response.choices[0].message.content


# Generate briefs for all six applications
briefs = {}

for letter_id, letter_text in LETTERS.items():

    row = extracted_df[
        extracted_df["letter_id"] == letter_id
    ].iloc[0]

    extracted_data = row.drop("letter_id").to_dict()

    briefs[letter_id] = generate_brief(
        letter_text,
        extracted_data
    )


# Print the three required examples
for letter_id in ["L001", "L002", "L006"]:
    print("=" * 70)
    print(f"DECISION-SUPPORT BRIEF — {letter_id}")
    print("=" * 70)
    print(briefs[letter_id])
    print()

DECISION-SUPPORT BRIEF — L001
## 1. Strengths
- The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market, indicating stability and familiarity with the market.
- She has a regular profit of GHS 900 each month from her current stall, showing a consistent income stream.
- Akosua has demonstrated savings discipline by accumulating GHS 2,500 over two years through the susu scheme without missing any contributions, which suggests she can manage regular payments.
- She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of security for the loan.

## 2. Risks / Red Flags
- The loan amount of GHS 8,000 is significant compared to her monthly profit of GHS 900, which might pose a risk if her expanded business does not generate enough additional income to cover the loan repayments.
- The repayment plan of GHS 450 monthly over 20 months is substantial and might strain her current cash flow, especially

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each?*

>**Answer:** Yes. The system should recognize that there are a number of strengths in L003, namely: there is a registered business, there are three apprentices, there is revenue and profit, there is a fixed deposit that can be pledged and there is a clear repayment proposal. The following are the main concerns for L006: The businesses have not yet inceptioned, the applicant wants a GHS 50,000 loan, there is no collaterals and the existing business income or monthly profit is not mentioned. The differences demonstrate that the system has the ability to pick out key information on widely varied risk profiles.

*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** Practically, an LLM can make mistakes or misunderstand information, so a human loan officer should review the evidence before making a financial decision. Keeping a human in the loop provides accountability and an opportunity to review or challenge the recommendation.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 7ce78a1f3ab787e197b600e19d328bdb6cadb60f

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [10]:
# Fields we need to evaluate
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]


def values_match(predicted, gold, field):
    # Handle applicant names case-insensitively
    if field == "applicant_name":
        if predicted is None or gold is None:
            return predicted == gold
        return str(predicted).strip().lower() == str(gold).strip().lower()

    # All other fields must match exactly
    return predicted == gold


evaluation_rows = []

for field in fields:

    row = {"field": field}

    correct_count = 0

    for letter_id in GOLD.keys():

        predicted_row = extracted_df[
            extracted_df["letter_id"] == letter_id
        ].iloc[0]

        predicted = predicted_row[field]
        gold_value = GOLD[letter_id][field]

        correct = values_match(
            predicted,
            gold_value,
            field
        )

        row[letter_id] = "✓" if correct else "✗"

        if correct:
            correct_count += 1

    row["accuracy"] = correct_count / len(GOLD)

    evaluation_rows.append(row)


accuracy_df = pd.DataFrame(evaluation_rows)

# Display as percentages
accuracy_df["accuracy"] = (
    accuracy_df["accuracy"] * 100
).round(1).astype(str) + "%"

display(accuracy_df)

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,100.0%
1,amount_ghs,✓,✓,✓,100.0%
2,purpose,✗,✗,✗,0.0%
3,monthly_profit_ghs,✓,✓,✗,66.7%
4,has_collateral_or_guarantor,✓,✓,✓,100.0%
5,repayment_months,✓,✓,✓,100.0%


### Part 4.2 — Reliability: is the system consistent?

In [11]:
def extract_fields_with_temperature(letter_text, temperature):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": EXTRACT_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": EXTRACT_PROMPT.format(
                    letter_text=letter_text
                )
            }
        ],
        temperature=temperature,
        max_tokens=300
    )

    result = response.choices[0].message.content.strip()

    # Remove markdown JSON fences if present
    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    if result.endswith("```"):
        result = result[:-3]

    result = result.strip()

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        return None


def reliability_test(temperature):
    results = []

    for i in range(5):
        result = extract_fields_with_temperature(
            LETTERS["L004"],
            temperature
        )
        results.append(result)

    valid_json = sum(
        result is not None
        for result in results
    )

    valid_results = [
        json.dumps(result, sort_keys=True)
        for result in results
        if result is not None
    ]

    unique_results = len(set(valid_results))

    identical = (
        valid_json == 5 and unique_results == 1
    )

    return results, valid_json, unique_results, identical


# Temperature 0.0
results_0, valid_0, unique_0, identical_0 = reliability_test(0.0)

# Temperature 1.0
results_1, valid_1, unique_1, identical_1 = reliability_test(1.0)


print("===== TEMPERATURE 0.0 =====")
print("Valid JSON:", valid_0, "/ 5")
print("Unique outputs:", unique_0)
print("All five identical:", identical_0)

print("\n===== TEMPERATURE 1.0 =====")
print("Valid JSON:", valid_1, "/ 5")
print("Unique outputs:", unique_1)
print("All five identical:", identical_1)

===== TEMPERATURE 0.0 =====
Valid JSON: 5 / 5
Unique outputs: 2
All five identical: False

===== TEMPERATURE 1.0 =====
Valid JSON: 5 / 5
Unique outputs: 3
All five identical: False


### Part 4.3 — Hallucination probing

In [12]:
# Test 1: Ask for information that is not provided
HALLUCINATION_PROMPT_1 = """
Read this loan application:

{letter_text}

What is the applicant's exact credit score?

If the credit score is not explicitly stated in the application,
say that it is not provided. Do not guess.
"""

answer_1, _ = ask_llm(
    HALLUCINATION_PROMPT_1.format(
        letter_text=LETTERS["L001"]
    ),
    temperature=0.0,
    max_tokens=150
)

print("===== HALLUCINATION TEST 1 =====")
print(answer_1)


# Test 2: Ask for a decision based on unsupported information
HALLUCINATION_PROMPT_2 = """
Read this loan application:

{letter_text}

The applicant has apparently defaulted on three previous loans.
Based on this information, should the loan be rejected?

Only use information that is actually stated in the application.
If the claim about previous defaults is not supported by the
application, explicitly say so and do not use it in your reasoning.
"""

answer_2, _ = ask_llm(
    HALLUCINATION_PROMPT_2.format(
        letter_text=LETTERS["L001"]
    ),
    temperature=0.0,
    max_tokens=200
)

print("\n===== HALLUCINATION TEST 2 =====")
print(answer_2)

===== HALLUCINATION TEST 1 =====
The applicant's credit score is not provided.

===== HALLUCINATION TEST 2 =====
The information about previous defaults is not mentioned in the application. Therefore, I will not consider it in my reasoning.

Based on the information provided in the application, the applicant has a stable business with a monthly profit of GHS 900, has saved GHS 2,500 with the susu scheme, and has a repayment plan of GHS 450 per month for 20 months. The applicant also has a guarantor, which provides an additional layer of security.

There is no information in the application that suggests the loan should be rejected. The applicant's proposal seems reasonable, and they have provided a clear plan for repayment. Therefore, based on the information provided in the application, there is no apparent reason to reject the loan.


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*

>**Answer:**
The extraction accuracy was high overall, showing that the model was able to identify most of the required information from the loan applications. The hardest field was 0.0%. This field was more difficult because the information was either expressed less directly in the application or was missing in some cases, requiring the model to correctly return null rather than make an assumption.
*2. What did the reliability experiment show about temperature and production systems?*

>**Answer:**

The reliability experiment showed that the system was more consistent at temperature 0.0 than at temperature 1.0. At temperature 0.0, all 5 outputs were valid JSON and there were 2 unique outputs. At temperature 1.0, all 5 outputs were also valid JSON, but there were 4 unique outputs. Neither temperature produced five completely identical outputs.

This shows that increasing the temperature increased the variation in the model's responses. However, even at temperature 0.0, the outputs were not completely identical. This is why structured extraction systems should use controlled, low-temperature generation and still validate the model's output before using it.

*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
Yes. The system showed some hallucination risk under probing, particularly when asked about information that was not supported by the loan application. The risk could be reduced by explicitly instructing the model to use only information in the source document, return “not provided” or null when information is missing, and never guess. A human loan officer should also review the output before making any final decision.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**

Poor English writing skills may lead to an unfair treatment as the AI might fail to correctly read the Applicant's application, even if the person is successful in their business. This could make the system unfair to people based on their communication skills rather than their actual financial situation.

Privacy and security issues arise when personal loan data is shared with another country's API. Prior to deploying I would look into the provider's data protection, security, data storage, retention policies and understand whether they would be applicable in Ghana or not.

a). The conservation of indices.b). The conservation of the flow of values.

Human review: A trained loan officer should review the AI's output before making any final decision.
Logging and monitoring: The system should have some sort of log of its outputs and it should be monitored for hallucination, errors, and unfair patterns.

a---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**

1. Prompting as engineering:
Both involve changing something and checking if the model performs better. Lab 3 focused on model hyperparameters, while Lab 4 focused on changing the instructions we give the LLM.

2. Trust:
I wouldn't trust it to run unattended. The biggest concern was that even at temperature 0.0, we got 2 different outputs from 5 runs.

3. Cost and scale:
I would take the average tokens from response.usage and multiply it by 1,000. This helps estimate the monthly cost and choose a provider with suitable pricing and limits.

4. Looking back at the course:
An API makes more sense because the model is already trained, so we don't need a huge dataset or the time and resources needed to train one ourselves. Training our own model would make more sense if we had a large specialized dataset and needed more control.A

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.